In [1]:
import sys, os
from pathlib import Path

# Cari root proyek
base = Path(os.getcwd())
ROOT = base
for pp in [base, base.parent, base.parent.parent]:
    if (pp / "data").exists():
        ROOT = pp
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
import src.config as config
from src.preprocess import prepare_data
from src.model import TSCP

DEVICE = "cpu"
print("Root:", ROOT)
print("Device:", DEVICE)

# Load data
data = prepare_data()
print(f"Train: {len(data['train'])} | Dev: {len(data['dev'])} | Test: {len(data['test'])}")
print(f"Vocab size: {len(data['word2idx'])}")

Root: C:\Users\Resky\Documents\PETE\Restaurant-Assistant-Chatbot-masters
Device: cpu
Data split -> Train: 405, Dev: 135, Test: 136
Samples -> Train: 1635, Dev: 553, Test: 556
Vocabulary size: 755 (|V| di paper: ~800)
Train: 1635 | Dev: 553 | Test: 556
Vocab size: 755


In [2]:
print("="*60)
print("HYPERPARAMETER RL YANG DIPAKAI:")
print("="*60)
print(f"RL_LEARNING_RATE       : {config.RL_LEARNING_RATE}")
print(f"RL_EPOCHS              : {config.RL_EPOCHS}")
print(f"RL_LAMBDA              : {config.RL_LAMBDA}")
print(f"RL_REWARD_POS          : {config.RL_REWARD_POS}")
print(f"RL_REWARD_NEG          : {config.RL_REWARD_NEG}")
print(f"RL_REWARD_HALLUCINATION: {config.RL_REWARD_HALLUCINATION}")
print(f"RL_SUPERVISED_ALPHA    : {config.RL_SUPERVISED_ALPHA}")
print("="*60)

HYPERPARAMETER RL YANG DIPAKAI:
RL_LEARNING_RATE       : 0.0001
RL_EPOCHS              : 15
RL_LAMBDA              : 0.8
RL_REWARD_POS          : 1.0
RL_REWARD_NEG          : -0.1
RL_REWARD_HALLUCINATION: -0.1
RL_SUPERVISED_ALPHA    : 0.5


In [3]:
import os

ckpt_paths = [
    os.path.join(config.SAVE_DIR, "tscp_rl_best.pt"),
    os.path.join(config.SAVE_DIR, "tscp_rl_final.pt"),
]

for path in ckpt_paths:
    if os.path.exists(path):
        os.remove(path)
        print(f"✅ Dihapus: {path}")
    else:
        print(f"⚠️ Tidak ada: {path}")

print("\n✅ Checkpoint RL lama sudah dihapus. Training baru akan membuat checkpoint baru.")

✅ Dihapus: C:\Users\Resky\Documents\PETE\Restaurant-Assistant-Chatbot-masters\src\..\checkpoints\tscp_rl_best.pt
✅ Dihapus: C:\Users\Resky\Documents\PETE\Restaurant-Assistant-Chatbot-masters\src\..\checkpoints\tscp_rl_final.pt

✅ Checkpoint RL lama sudah dihapus. Training baru akan membuat checkpoint baru.


In [4]:
import torch
from src.model import TSCP
from src.train_rl import train_rl

# Load model supervised sebagai starting point
ckpt_path = os.path.join(config.SAVE_DIR, "tscp_supervised_best.pt")
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
print(f"Loaded supervised model from epoch {ckpt['epoch']} (dev_loss={ckpt['dev_loss']:.4f})")

# Inisialisasi model
vocab_size = len(data['word2idx'])
model = TSCP(vocab_size).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print("✅ Model loaded successfully!")

# JALANKAN RL TRAINING (INI YANG SEBELUMNYA TIDAK ADA!)
print("\n" + "="*60)
print("MEMULAI RL FINE-TUNING...")
print("="*60)
model = train_rl(model, data, device=DEVICE)
print("\n✅ RL training selesai!")

Loaded supervised model from epoch 16 (dev_loss=1.5780)
✅ Model loaded successfully!

MEMULAI RL FINE-TUNING...

REINFORCEMENT LEARNING FINE-TUNING
Learning rate: 0.0001
Lambda (decay): 0.8
Reward: +1.0 (slot match), -0.1 (slot halusinasi), -0.1 (token normal)
Supervised anchor alpha: 0.5
Max epochs: 15
Train samples dengan request slot: 480 / 1635
Baseline dev Success-F1 (sebelum RL): 0.6098
RL Epoch   1/15 | Avg Loss: 0.1528 | Avg Reward: -0.4119 | Dev Success-F1: 0.8492 | Time: 48.1s
  → Best RL model saved (dev Success-F1=0.8492)
RL Epoch   2/15 | Avg Loss: 0.1343 | Avg Reward: -0.1131 | Dev Success-F1: 0.9372 | Time: 44.4s
  → Best RL model saved (dev Success-F1=0.9372)
RL Epoch   3/15 | Avg Loss: 0.1230 | Avg Reward: -0.0260 | Dev Success-F1: 0.9372 | Time: 43.4s
RL Epoch   4/15 | Avg Loss: 0.1333 | Avg Reward: 0.1502 | Dev Success-F1: 0.8093 | Time: 41.8s
RL Epoch   5/15 | Avg Loss: 0.1034 | Avg Reward: 0.1831 | Dev Success-F1: 0.8827 | Time: 43.9s
RL Epoch   6/15 | Avg Loss: 0.

In [5]:
from src.evaluate import evaluate_model

print("\n" + "="*60)
print("EVALUASI MODEL SETELAH RL FINE-TUNING")
print("="*60)

results = evaluate_model(model, data['test'], data['word2idx'], data['idx2word'], 
                         data['database'], device=DEVICE)

print("\n" + "="*60)
print("PERBANDINGAN HASIL")
print("="*60)
print(f"{'Metrik':<25} {'Baseline (Supervised)':<25} {'RL Fine-Tuned':<25}")
print("-"*60)
print(f"{'BLEU':<25} {'0.1671':<25} {results['BLEU']:.4f}")
print(f"{'Entity Match Rate':<25} {'0.8201':<25} {results['Entity_Match_Rate']:.4f}")
print(f"{'Success F1':<25} {'0.5753':<25} {results['Success_F1']:.4f}")
print("="*60)


EVALUASI MODEL SETELAH RL FINE-TUNING

EVALUATING ON TEST SET (556 samples)

--- Sample 0 ---
  Input:          i'm looking for a moderately priced restaurant serving cuban food....
  Gold Bspan:     <inf> cuban ; moderate </inf> <req>  </req>
  Pred Bspan:     <inf> moderate </inf> <req>  </req>
  Gold Response:  i'm sorry we do not have any cuban restaurants in the PRICERANGE_SLOT price rang...
  Pred Response:  NAME_SLOT is a FOOD_SLOT restaurant in the AREA_SLOT part of town ....

--- Sample 1 ---
  Input:          <inf> cuban ; moderate </inf> <req>  </req> i'm sorry we do not have any cuban r...
  Gold Bspan:     <inf> british ; moderate </inf> <req>  </req>
  Pred Bspan:     <inf> british ; moderate </inf> <req>  </req>
  Gold Response:  NAME_SLOT serves FOOD_SLOT food. would you like more information about this loca...
  Pred Response:  NAME_SLOT is in the PRICERANGE_SLOT price range ....

--- Sample 2 ---
  Input:          <inf> british ; moderate </inf> <req>  </req> NAME_SL